# Policy Gradient & Actor-Critic

**Companion lesson:** https://ml-viz.vercel.app/courses/reinforcement-learning/04-policy-gradient

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## REINFORCE, from scratch

Instead of values, optimize the **policy** directly. A softmax policy over per-state logits; after each episode we push up the log-probability of actions weighted by the (baseline-subtracted) return.

In [ ]:
class GridWorld:
    """n x n grid. Start top-left (0), goal bottom-right. Actions: 0=up 1=down 2=left 3=right.
    Reward -1 per step, +10 at the goal (terminal)."""
    def __init__(self, n=5):
        self.n = n; self.nS = n*n; self.nA = 4; self.goal = n*n-1
    def step(self, s, a):
        r, c = divmod(s, self.n)
        if a==0: r = max(0, r-1)
        elif a==1: r = min(self.n-1, r+1)
        elif a==2: c = max(0, c-1)
        else: c = min(self.n-1, c+1)
        s2 = r*self.n + c
        done = (s2 == self.goal)
        return s2, (10.0 if done else -1.0), done

env = GridWorld(5)

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e/e.sum()
print('policy = softmax over 4 actions in each of', env.nS, 'states')

## The policy gradient update

$\nabla_\theta J = \mathbb{E}[\nabla_\theta\log\pi(a|s)\,(G_t-b)]$. For a softmax, $\nabla_{\theta_{s,a'}}\log\pi(a|s) = \mathbb{1}[a'=a]-\pi(a'|s)$.

In [ ]:
def reinforce(env, episodes=3000, lr=0.1, gamma=0.95):
    rng = np.random.RandomState(0)
    theta = np.zeros((env.nS, env.nA))     # policy logits
    returns = []
    for ep in range(episodes):
        s, done, traj, t = 0, False, [], 0
        while not done and t < 100:
            p = softmax(theta[s]); a = rng.choice(env.nA, p=p)
            s2, r, done = env.step(s, a); traj.append((s,a,r)); s=s2; t+=1
        # discounted returns-to-go
        G, Gs = 0, []
        for (_,_,r) in reversed(traj):
            G = r + gamma*G; Gs.insert(0, G)
        Gs = np.array(Gs); baseline = Gs.mean()
        for (s,a,_), G in zip(traj, Gs):
            p = softmax(theta[s]); grad = -p; grad[a] += 1     # d log pi / d theta
            theta[s] += lr * (G - baseline) * grad
        returns.append(sum(r for _,_,r in traj))
    return theta, returns

theta, returns = reinforce(env)
print('trained policy over', len(returns), 'episodes')

## The policy improves; the advantage sign drives it

A return above the baseline (positive advantage) makes the taken action more likely; below makes it less likely.

In [ ]:
ma = np.convolve(returns, np.ones(100)/100, mode='valid')
fig, ax = plt.subplots(1, 2, figsize=(13,5))
ax[0].plot(ma, color='#f59e0b'); ax[0].set_xlabel('episode'); ax[0].set_ylabel('return (100-ep avg)')
ax[0].set_title('REINFORCE on the gridworld')
arrows = {0:'↑',1:'↓',2:'←',3:'→'}
pi = theta.argmax(1)
ax[1].imshow(theta.max(1).reshape(5,5), cmap='magma')
for s in range(env.nS):
    r,c = divmod(s, env.n)
    ax[1].text(c, r, 'G' if s==env.goal else arrows[pi[s]], ha='center', va='center', color='w', fontsize=16)
ax[1].set_title('learned policy (greedy action)'); ax[1].axis('off'); plt.show()

## Key takeaways

- Policy gradient optimizes $\pi_\theta$ directly — no value table needed to act.
- The update raises log-probability of actions weighted by their **return**.
- Subtracting a **baseline** (the mean return) cuts variance — the seed of actor-critic.
- It naturally handles stochastic policies and, unlike DQN, continuous actions.